# CatVTON - Virtual Try-On API Server
Run this notebook top-to-bottom to start the Virtual Try-On API.

In [ ]:
# ==========================================
# 1. SETUP ENVIRONMENT & DEPENDENCIES
# (This will take ~2-3 minutes)
# ==========================================
import os

print("1/3: Cloning CatVTON Repository...")
if not os.path.exists('/content/CatVTON'):
    !git clone https://github.com/Zheng-Chong/CatVTON.git > /dev/null 2>&1

print("2/3: Creating clean Virtual Environment...")
!python3.10 -m venv --without-pip /content/vtoenv
!curl -sS https://bootstrap.pypa.io/get-pip.py | /content/vtoenv/bin/python > /dev/null 2>&1

print("3/3: Installing Dependencies (Please wait)...")
PY = "/content/vtoenv/bin/python"

!$PY -m pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!$PY -m pip install -q diffusers==0.30.3 accelerate==0.29.2 transformers==4.46.3 huggingface_hub==0.25.2
!$PY -m pip install -q matplotlib==3.9.1 numpy==1.26.4 opencv-python==4.10.0.84 pillow==10.3.0 PyYAML==6.0.1 scipy==1.13.1 setuptools==69.5.1 scikit-image==0.24.0 tqdm==4.66.4 fvcore==0.1.5.post20221221 cloudpickle==3.0.0 omegaconf==2.3.0 pycocotools==2.0.8 av==12.3.0 gradio==4.41.0 peft==0.17.0 fastapi uvicorn python-multipart flask flask-cors
!$PY -m pip install -q matplotlib-inline

# Download Cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("✅ Setup complete!")

In [ ]:
%%writefile /content/CatVTON/api_server.py
# ==========================================
# 2. API SERVER SCRIPT
# ==========================================
import os, io, uuid, base64
from flask import Flask, request, jsonify
import matplotlib
matplotlib.use('Agg')

try:
    from app import pipeline, automasker, args, resize_and_crop, resize_and_padding
    from PIL import Image
    print("✅ CatVTON Pipeline successfully imported!")
except ImportError as e:
    print("❌ Warning: Could not import CatVTON pipeline.")
    print(e)
    pipeline = None

app = Flask(__name__)
API_KEY = "catvton_BsNYhpfJXLzS7wG3ghNE3v64nVdqL6o4azhrcQTAEz8"

@app.route("/", methods=["GET"])
def home():
    return jsonify({"status": "online", "service": "CatVTON Inference API"})

@app.route("/tryon", methods=["POST"])
def tryon():
    if request.headers.get("X-API-Key") != API_KEY:
        return jsonify({"error": "Invalid API key"}), 401
    
    if "person" not in request.files or "cloth" not in request.files:
        return jsonify({"error": "Missing images"}), 400

    try:
        person_image = Image.open(request.files["person"].stream).convert("RGB")
        cloth_image = Image.open(request.files["cloth"].stream).convert("RGB")

        if pipeline is None: return jsonify({"error": "Pipeline not loaded"}), 500

        person_image = resize_and_crop(person_image, (args.width, args.height))
        cloth_image = resize_and_padding(cloth_image, (args.width, args.height))
        mask = automasker(person_image, "upper_body")['mask']

        result_image = pipeline(
            image=person_image, condition_image=cloth_image, mask=mask,
            num_inference_steps=30, guidance_scale=2.5
        )[0]

        buffered = io.BytesIO()
        result_image.save(buffered, format="JPEG", quality=90)
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
        
        return jsonify({"success": True, "job_id": str(uuid.uuid4()), "image": f"data:image/jpeg;base64,{img_str}"})

    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=7860)

In [ ]:
# ==========================================
# 3. LAUNCH SERVER & CLOUDFLARE TUNNEL
# ==========================================
import subprocess
import time
import re

# Kill any stuck processes on port 7860
!fuser -k 7860/tcp > /dev/null 2>&1
time.sleep(2)

print("🚀 Loading AI Models (Takes ~30 seconds)...")
server_process = subprocess.Popen(
    ["/content/vtoenv/bin/python", "/content/CatVTON/api_server.py"],
    cwd="/content/CatVTON",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("🌐 Starting Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(15):
    line = tunnel_process.stdout.readline()
    matches = re.search(r"(https://[a-zA-Z0-9.-]+\.trycloudflare\.com)", line)
    if matches:
        public_url = matches.group(1)
        break
    time.sleep(1)

print("\n" + "="*70)
if public_url:
    print("🎉 EVERYTHING IS ONLINE!")
    print(f"👉 UPDATE YOUR BACKEND .env WITH THIS URL: {public_url}")
else:
    print("❌ Cloudflare tunnel failed to start. Run this cell again.")
print("="*70 + "\n")

print("--- LIVE SERVER LOGS ---")
try:
    while True:
        line = server_process.stdout.readline()
        if line:
            print(line.strip())
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Stopping servers...")
    server_process.kill()
    tunnel_process.kill()
    print("Stopped.")